DMP PDF
  ↓
1. pdfplumber extraction
  - extract line-level text
  - keep page number, font size, bold, position

  ↓
2. rule-based structure detection
  - Element 1, Element 2 → section
  - A., B., C. → question
  - remaining lines → answer/content

  ↓
3. build narrative JSON
  - load your full RDA + DMPTool extension skeleton
  - keep metadata fields as null
  - fill only narrative.template.section

  ↓
4. save final JSON
  - output looks like your skeleton
  - narrative part contains extracted DMP sections/questions/answers

PDF line label              JSON location
--------------------------------------------------
section                     narrative.template.section[].title

subsection                  narrative.template.section[].question[].text

content after subsection    question[].answer.json.answer[].text

In [25]:
from pathlib import Path
import pandas as pd

from dmpbridge.pdf.pdfplumber_extractor import save_pdfplumber_outputs
from dmpbridge.processing.structure_detector import detect_structure
from dmpbridge.processing.structure_json_builder import save_narrative_json

In [26]:
project_root = Path.cwd().parent

pdf_path = project_root / "data" / "raw_pdfs" / "sample2.pdf"
skeleton_path = project_root / "schemas" / "rda_dmp_dmptool_extension_skeleton.json"

pdfplumber_json_path = project_root / "data" / "pdfplumber_blocks" / f"{pdf_path.stem}.json"
csv_output_path = project_root / "outputs" / "debug" / f"{pdf_path.stem}_structured_lines.csv"
final_json_path = project_root / "data" / "structure_json" / f"{pdf_path.stem}_narrative.json"

print("PDF exists:", pdf_path.exists())
print("Skeleton exists:", skeleton_path.exists())

PDF exists: True
Skeleton exists: True


In [27]:
blocks = save_pdfplumber_outputs(pdf_path)

print("Extracted lines:", len(blocks))
print("Saved pdfplumber JSON:", pdfplumber_json_path.exists())

[2026-05-05 10:17:29] Extracting line-level text with pdfplumber: sample2.pdf
[2026-05-05 10:17:30] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_blocks\sample2.json
[2026-05-05 10:17:30] Saved extracted text: C:\Users\Nahid\dmpbridge\data\extracted_text\sample2.txt
Extracted lines: 171
Saved pdfplumber JSON: True


In [28]:

structured_blocks = detect_structure(blocks)

df = pd.DataFrame(structured_blocks)
print("Detected format:", df["document_format"].iloc[0])
print(df["label"].value_counts())

df[[
    "page",
    "line_order",
    "text",
    "avg_font_size",
    "is_bold",
    "label",
    "document_format"
]].head(100)

Detected format: numbered_sections
label
content           166
section             4
document_title      1
Name: count, dtype: int64


,page,line_order,text,avg_font_size,is_bold,label,document_format
0,1,1,Center for Bio-Inspired Energy Science,13.199999,True,document_title,numbered_sections
1,1,2,1. Data sharing and preservation,13.199999,True,section,numbered_sections
2,1,3,Data management plans should describe whether ...,12.097499,True,content,numbered_sections
3,1,4,of the proposed research will be shared and pr...,12.097499,True,content,numbered_sections
4,1,5,"preserve certain data, then the plan must expl...",12.097499,True,content,numbered_sections
...,...,...,...,...,...,...,...
95,3,12,greatest extent and with the fewest constraint...,10.994999,False,content,numbered_sections
96,3,13,"principles of this Statement, data sharing sho...",10.994999,False,content,numbered_sections
97,3,14,"the scientific community, industry, and the pu...",10.994999,False,content,numbered_sections
98,3,15,these guidelines.,10.994999,False,content,numbered_sections


In [29]:
print("Detected document format:", df["document_format"].iloc[0])
print(df["label"].value_counts())

Detected document format: numbered_sections
label
content           166
section             4
document_title      1
Name: count, dtype: int64


In [30]:
csv_output_path.parent.mkdir(parents=True, exist_ok=True)

df[[
    "page",
    "line_order",
    "text",
    "avg_font_size",
    "is_bold",
    "label",
    "document_format"
]].to_csv(csv_output_path, index=False, encoding="utf-8")

print("Saved CSV:", csv_output_path)

Saved CSV: c:\Users\Nahid\dmpbridge\outputs\debug\sample2_structured_lines.csv


In [31]:
final_json = save_narrative_json(
    structured_blocks=structured_blocks,
    output_path=final_json_path,
    skeleton_path=skeleton_path
)

sections = final_json["narrative"]["template"]["section"]

print("Saved JSON:", final_json_path)
print("Number of sections:", len(sections))

for sec in sections:
    print(sec["order"], sec["title"], "| questions:", len(sec["question"]))

[2026-05-05 10:17:30] Saved narrative JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample2_narrative.json
Saved JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample2_narrative.json
Number of sections: 4
1 1. Data sharing and preservation | questions: 0
2 2. Data used in publications | questions: 0
3 3. Data management resources | questions: 0
4 4. Confidentiality, security and rights | questions: 0


In [32]:
sections[0]

{'id': 'section_1',
 'title': '1. Data sharing and preservation',
 'description': 'Data management plans should describe whether and how data generated in the course\nof the proposed research will be shared and preserved. If the plan is not to share and/or\npreserve certain data, then the plan must explain the basis of the decision (for example,\ncost/benefit considerations, other parameters of feasibility, scientific appropriateness, or\nlimitations discussed in #4). At a minimum, DMPs must describe how data sharing and\npreservation will enable validation of results, or how results could be validated if data\nare not shared or preserved.\nRoles & Responsibilities. For the proposed research, Director Samuel Stupp with help from the\nExecutive Director of Research will take the lead and responsibility for coordinating and ensuring data\nstorage and access and communicating expectations to all investigators. However, all senior\ninvestigators will also be involved in managing, storing, 